In [9]:
import os
import numpy as np
import xarray as xr
import capytaine as cpt
from capytaine.io.legacy import export_hydrostatics

In [10]:
def run_sphere(x,y):
    input_data_dir = os.getcwd()
    output_dir = os.path.join(input_data_dir, f'outputs_x{str(x)}y{str(y)}.nc')
    os.makedirs(output_dir, exist_ok=True)

    mesh_file = os.path.join(input_data_dir, "sphere.dat")
    mesh = cpt.load_mesh(mesh_file, file_format="nemoh")
    mesh = mesh.translate_x(x)
    mesh = mesh.translate_y(y)
    # Symmetry defined in header of "sphere.dat" is used.
    body = cpt.FloatingBody(
        mesh=mesh,
        lid_mesh=mesh.generate_lid(),
        dofs=cpt.rigid_body_dofs(rotation_center=(0, 0, -2.0)),
        center_of_mass=(0, 0, -2.0),
        name="floating_sphere"
    )

    body.inertia_matrix = body.compute_rigid_body_inertia()
    body.hydrostatic_stiffness = body.immersed_part().compute_hydrostatic_stiffness()

    #body.show()  # Uncomment to display the mesh in 3D for verification
    #body.show_matplotlib()

    test_matrix = xr.Dataset(coords={
        "omega": np.linspace(0.02, 8.4, 420),
        "radiating_dof": list(body.dofs),
        "wave_direction": [0],
        "water_depth": [50.0],
        "rho": [1000.0],
        })

    solver = cpt.BEMSolver()
    dataset = solver.fill_dataset(test_matrix, body.immersed_part(), n_jobs=1)

    cpt.export_dataset(os.path.join(output_dir, f'sphere_x{str(x)}y{str(y)}.nc'), dataset)
    export_hydrostatics(output_dir, body)
    


In [11]:
# Create the grid of x and y coordinates
x_values = np.arange(-2, 2.5, 0.5)  # 2.5 to include 2 in the range
y_values = np.arange(-2, 2.5, 0.5)

# Create a meshgrid
X, Y = np.meshgrid(x_values, y_values)

# Flatten the arrays to iterate through each combination
x_flat = X.flatten()
y_flat = Y.flatten()

# Iterate through each combination of x and y
for x, y in zip(x_flat, y_flat):
    run_sphere(x,y)


[14:29:48] WARNING  Mesh resolution for 889 problems:                                                              
                    The resolution of the mesh might be insufficient for omega ranging from 5.880 to 8.400.        
                    This warning appears when the largest panel of this mesh has radius > wavelength/8.

c:\Users\jtgrasb\AppData\Local\anaconda3\envs\capy_WS\Lib\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

KeyboardInterrupt: 

Output()